# 07 — PDFs, temas e evidência para enriquecimento

Extrai no máximo as três primeiras páginas de até 18 documentos, sem salvar os PDFs. `pypdf` é opcional: se faltar, o check fica inconclusivo. Rótulos candidatos **não são publicados**; todos requerem revisão humana, inclusive direção de medida.

In [1]:
from pathlib import Path
import sys
import pandas as pd
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'validation_utils.py').exists())
sys.path.insert(0, str(root / 'scripts'))
from validation_utils import (Probe, OUT, CHECKS, MANIFEST, MANIFEST_FIELDS, api_camara, api_senado,
                              record, manifest_rows, save_manifest, rows_camara, count_values, csv_head)
pd.set_option('display.max_colwidth', 90)

import csv, hashlib, importlib.util, json
probe=Probe('07',max_calls=90)
manifest=manifest_rows()
assert manifest, 'Execute 03 primeiro.'
pdf_available=bool(importlib.util.find_spec('pypdf'))
print('pypdf disponível:',pdf_available)

pypdf disponível: True


## Seleção de documentos e versões

In [2]:
import re
existing=list(csv.DictReader((root/'docs/amostra_projetos.csv').open(encoding='utf-8-sig',newline='')))
selected=existing[:12]
# Add up to one historical project per Casa/year. Missing URL remains explicit.
historical=[]
for house in ('camara','senado'):
  for year in ('2023','2024','2025'):
    item=next((x for x in manifest if x['entidade']=='projeto' and x['fonte']==house and x['ano']==year),None)
    if not item: continue
    detail=probe.json(item['url'])
    url=''
    if detail.ok and house=='camara': url=(detail.data.get('dados') or {}).get('urlInteiroTeor') or ''
    if detail.ok and house=='senado':
        doc=probe.json(api_senado('processo/documento.json'),{'idProcesso':item['id']})
        docs_data=doc.data if doc.ok and isinstance(doc.data,list) else []
        url=next((d.get('urlDocumento') for d in docs_data if d.get('urlDocumento')),'')
    historical.append({'fonte':house,'identificador':item['id'],'tipo':item['tipo'],
                       'url_documento':url,'temas_oficiais':'[]','ano_amostra':year})
selected += historical
docs=[]; snippets=[]
for row in selected:
    url=row['url_documento']
    r=probe.fetch(url,limit=15_000_000) if url else None
    pages=text_chars=0; evidence=''; error=r.error if r else 'sem URL'
    if r and r.ok and pdf_available:
        try:
            from pypdf import PdfReader
            from io import BytesIO
            reader=PdfReader(BytesIO(r.content))
            pages=len(reader.pages)
            page_texts=[reader.pages[i].extract_text() or '' for i in range(min(3,pages))]
            text_chars=sum(map(len,page_texts))
            evidence=hashlib.sha256(r.content).hexdigest()[:16]
            for page_num,text_page in enumerate(page_texts,1):
                for pattern,topic in ((r'meio ambiente|ambiental|floresta','ambiente e clima'),
                                      (r'saúde|hospital|medicamento','saúde'),
                                      (r'tribut|imposto|contribuição','economia e tributos'),
                                      (r'trabalho|emprego|salário','trabalho')):
                    match=re.search(pattern,text_page,re.IGNORECASE)
                    if match:
                        snippet=' '.join(text_page[max(0,match.start()-55):match.end()+90].split())[:170]
                        snippets.append({'fonte':row['fonte'],'id_projeto':row['identificador'],'url_documento':r.url,
                                         'pagina':page_num,'trecho':snippet,'tema_comum_candidato':topic,
                                         'dimensao':'','direcao':'indeterminado','status_revisao':'pendente',
                                         'motivo':'Palavra-chave localizou trecho; contexto e proposta ainda não revisados'})
                        break
                if any(s['id_projeto']==row['identificador'] for s in snippets): break
        except Exception as exc: error=f'extração: {type(exc).__name__}: {exc}'
    docs.append({'fonte':row['fonte'],'id':row['identificador'],'tipo':row['tipo'],
                 'ano':row.get('ano_amostra','2026'),
                 'HTTP':r.status if r else None,'mime':(r.headers or {}).get('Content-Type') if r else None,
                 'bytes':len(r.content or b'') if r else 0,'paginas':pages,'caracteres_3_paginas':text_chars,
                 'hash_prefixo':evidence,'url':r.url if r else url,'erro':error})
display(pd.DataFrame(docs).drop(columns=['url','erro']))
record('07','pdfs','ambas','URLs oficiais do manifesto 2026',len(docs),
       'HTTP/PDF, limite, versão/hash e texto por página medidos sem assumir extração universal',
       {'acessiveis':sum(x['HTTP']==200 for x in docs),'texto_extraivel':sum(x['caracteres_3_paginas']>0 for x in docs),
        'anos':sorted(set(x['ano'] for x in docs)),
        'falhas':[x['id'] for x in docs if x['erro']]},
       'PASSOU' if pdf_available and all(x['HTTP']==200 and x['caracteres_3_paginas']>0 for x in docs) else 'INCONCLUSIVO',
       'Páginas sem texto e PDFs históricos ainda exigem amostra; pypdf ausente' if not pdf_available else '')

,fonte,id,tipo,ano,HTTP,mime,bytes,paginas,caracteres_3_paginas,hash_prefixo
0,camara,2647384,PL,2026,200,NaN,107422,3,5626,a6166810e306c5e4
1,camara,2647345,PL,2026,200,NaN,123346,11,6828,0d555a5854f0ca0f
2,camara,2599994,PLP,2026,200,NaN,158536,4,6150,eacf92f4040189fa
3,camara,2600175,PLP,2026,200,NaN,117939,3,5859,1bc96dd79b30e99d
4,camara,2604173,PEC,2026,200,NaN,143434,5,5823,af8587ffad75ec1a
5,camara,2623400,PEC,2026,200,NaN,143785,8,5217,72d20b888526d88b
6,senado,9101538,PL,2026,200,application/pdf,99987,5,4047,84614bf343755d90
7,senado,9101917,PL,2026,200,application/pdf,124379,7,6763,ffb43c134a32c8ef
8,senado,9104876,PLP,2026,200,application/pdf,165403,9,6087,9064c566522ce57c
9,senado,9105948,PLP,2026,200,application/pdf,350038,3,5142,1f31f20ed5fa8b4a


{'notebook': '07',
 'check': 'pdfs',
 'fonte': 'ambas',
 'url_parametros': 'URLs oficiais do manifesto 2026',
 'coletado_em_utc': '2026-10-09T14:23:57+00:00',
 'amostra_n': 18,
 'regra': 'HTTP/PDF, limite, versão/hash e texto por página medidos sem assumir extração universal',
 'contagens': '{"acessiveis": 18, "anos": ["2023", "2024", "2025", "2026"], "falhas": [], "texto_extraivel": 18}',
 'estado': 'PASSOU',
 'motivo': ''}

## Temas oficiais faltantes e candidatos sem classificação publicada

In [3]:
theme_rows=[{'fonte':x['fonte'],'tipo':x['tipo'],'id':x['identificador'],
             'temas':json.loads(x['temas_oficiais'] or '[]')} for x in existing]
display(pd.DataFrame([{'fonte':x['fonte'],'tipo':x['tipo'],'id':x['id'],'n_temas':len(x['temas'])} for x in theme_rows]))
candidate_path=OUT/'candidatos_rotulos.csv'
with candidate_path.open('w',encoding='utf-8-sig',newline='') as f:
    w=csv.DictWriter(f,fieldnames=['fonte','id_projeto','url_documento','pagina','trecho','tema_comum_candidato',
                                    'dimensao','direcao','status_revisao','motivo'])
    w.writeheader()
    w.writerows(snippets)
display(pd.DataFrame(snippets)[['fonte','id_projeto','pagina','tema_comum_candidato','trecho']] if snippets else pd.DataFrame())
record('07','temas_e_rotulos','ambas',str(candidate_path),len(theme_rows),
       'Preservar tema oficial; rótulo derivado só com documento, página, trecho e revisão',
       {'temas_faltantes':sum(not x['temas'] for x in theme_rows),'candidatos_para_revisao':len(snippets),'rotulos_publicados':0},
       'INCONCLUSIVO','Candidatos aguardam leitura e revisão; não inferir direção de ementa ou palavras-chave')

,fonte,tipo,id,n_temas
0,camara,PL,2647384,0
1,camara,PL,2647345,0
2,camara,PLP,2599994,3
3,camara,PLP,2600175,2
4,camara,PEC,2604173,3
5,camara,PEC,2623400,3
6,senado,PL,9101538,2
7,senado,PL,9101917,4
8,senado,PLP,9104876,1
9,senado,PLP,9105948,2


,fonte,id_projeto,pagina,tema_comum_candidato,trecho
0,camara,2647384,1,trabalho,para trinta horas semanais a jornada máxima de trabalho dos profissionais do magistéri...
1,camara,2599994,1,economia e tributos,"14 de dezembro de 2006, para estabelecer tratamento tributário diferenciado aplicável ..."
2,camara,2600175,1,economia e tributos,"m da limitação da redução a zero de alíquotas do Imposto sobre Bens e Serviços (IBS), ..."
3,camara,2604173,1,economia e tributos,alíquota máxima e base de cálculo exclusiva do Imposto sobre a Propriedade de Veículos...
4,senado,9101538,1,saúde,"E 2026 Tipifica os crimes de desvio de recursos da saúde pública, fraude em registros ..."
5,senado,9101917,3,trabalho,"V – as parcelas vinculadas a condições específicas de trabalho, local, horário ou ativ..."
6,senado,9104876,1,economia e tributos,em a instituição de regime de isenção condicionada do Imposto sobre Transmissão Causa ...
7,senado,9105948,1,economia e tributos,"de março de 2026, para incluir o regime especial de tributação para serviços de datace..."
8,senado,8986903,1,economia e tributos,"do-a, para alguns setores da atividade econômica, por contribuição incidente sobre a r..."
9,senado,9104654,2,trabalho,III – o Presidente do Tribunal Superior do Trabalho; IV – o Presidente do Superior Tri...


{'notebook': '07',
 'check': 'temas_e_rotulos',
 'fonte': 'ambas',
 'url_parametros': 'G:\\Meu Drive\\Projetos pessoais\\Dashboard Lesgislativo BR\\docs\\validacao\\candidatos_rotulos.csv',
 'coletado_em_utc': '2026-10-09T14:23:57+00:00',
 'amostra_n': 12,
 'regra': 'Preservar tema oficial; rótulo derivado só com documento, página, trecho e revisão',
 'contagens': '{"candidatos_para_revisao": 13, "rotulos_publicados": 0, "temas_faltantes": 2}',
 'estado': 'INCONCLUSIVO',
 'motivo': 'Candidatos aguardam leitura e revisão; não inferir direção de ementa ou palavras-chave'}

## Ideologia partidária: disponibilidade não equivale a cobertura atual

In [4]:
sources={'Pesquisa Legislativa Brasileira':'https://doi.org/10.7910/DVN/WM9IZ8',
         'V-Party':'https://v-dem.net/data/v-party-dataset/'}
heads=[]
for name,url in sources.items():
    r=probe.fetch(url,method='HEAD')
    heads.append({'fonte':name,'HTTP':r.status,'URL_final':r.url,'erro':r.error})
display(pd.DataFrame(heads))
record('07','ideologia_atual','externa',' | '.join(sources.values()),len(heads),
       'Escala, ano, partidos atuais e correspondência partidária precisam ser verificáveis',
       {'fontes_acessiveis':sum(x['HTTP']==200 for x in heads),'partidos_atuais_mapeados':0},
       'INCONCLUSIVO','Acesso à página não comprova cobertura da 57ª legislatura; indicador desativado')

,fonte,HTTP,URL_final,erro
0,Pesquisa Legislativa Brasileira,202,https://dataverse.harvard.edu/citation?persistentId=doi:10.7910/DVN/WM9IZ8,HTTP 202
1,V-Party,200,https://v-dem.net/data/v-party-dataset/,


{'notebook': '07',
 'check': 'ideologia_atual',
 'fonte': 'externa',
 'url_parametros': 'https://doi.org/10.7910/DVN/WM9IZ8 | https://v-dem.net/data/v-party-dataset/',
 'coletado_em_utc': '2026-10-09T14:24:00+00:00',
 'amostra_n': 2,
 'regra': 'Escala, ano, partidos atuais e correspondência partidária precisam ser verificáveis',
 'contagens': '{"fontes_acessiveis": 1, "partidos_atuais_mapeados": 0}',
 'estado': 'INCONCLUSIVO',
 'motivo': 'Acesso à página não comprova cobertura da 57ª legislatura; indicador desativado'}